# Sweeping a parameter

One scenario per value of the unsecured arm's `premium`, over 200,000 applications in batches of 50,000.
The `Progress` callback reports after every batch, and no metric holds the whole input.

**How an experiment runs.** The runner feeds the *baseline* (the unchanged flow) and every *scenario* the
same records, one batch at a time. Each **metric** gets its own copy per scenario and collects through hooks:
a *global* metric reads the flow's output; a *step* metric (`nodes="<glob>"`) reads what matching steps
wrote, on the records that ran them. `summarise()` gives a typed summary; a **plot** draws one metric's
summaries across scenarios. `result.show()` lays it all out.

In [1]:
from decikit.experiments import *
from example_projects.experiments.loan_pipeline import applications, build

ENTRY = "example_projects.experiments.loan_pipeline:build"
pipeline = build()   # ratio -> rate -> pricing (secured | unsecured) -> approved, payment
loans = applications(200_000)
loans.head()

income,debt,collateral
f64,f64,f64
3829.0,718.0,1344.0
3454.0,938.0,3913.0
4704.0,1956.0,0.0
3797.0,1960.0,16821.0
2939.0,387.0,0.0


In [2]:
exp = Experiment("premium-sweep", pipeline, loans, entry=ENTRY, batch_size=50_000)
for premium in (0.02, 0.06, 0.08):
    exp.scenario(f"premium_{premium}", params={"loans": {"pricing": {"unsecured": {"premium": premium}}}})
exp.metric(
    Divergence(column="approved"),                                    # global
    ValueCounts(column="approved"),                                   # global
    Histogram(name="payment", column="payment", range=(0, 400)),      # global
    Histogram(name="unsecured_rate", column="rate", nodes="loans/pricing/unsecured", range=(0.1, 0.35)),
)
exp.plot(Changes(metric="divergence"), Distribution(metric="unsecured_rate"),
         Distribution(name="payments", metric="payment"))
exp.callback(Progress())
result = exp.run()

premium-sweep: 50,000/200,000 records, 0.4s
premium-sweep: 100,000/200,000 records, 0.4s
premium-sweep: 150,000/200,000 records, 0.4s
premium-sweep: 200,000/200,000 records, 0.5s


The unsecured arm's rate, binned the same way for every scenario; click a scenario in the legend to pick
it out. The pipeline's own premium is 0.04, so the baseline is the reference.

In [3]:
result.plots["unsecured_rate"].show()

<vega-lite chart>

In [4]:
result.show()